# 🌍 AI Stick — NLLB-200 Backend (Google Colab)

Runs the full Runyoro-Rutooro translation backend with **NLLB-200 (FP32, GPU)**  
and exposes it via a public `cloudflared` tunnel — no ngrok account needed.

### How to keep this running 24/7
1. **Run all cells** (Runtime → Run all)
2. The last cell starts a **keep-alive loop** inside the notebook that:
   - Pings the tunnel URL every 4 minutes so Colab doesn't see it as idle
   - Runs a JavaScript trick that clicks the "Stay connected" button automatically
   - Reconnects if the tunnel drops
3. Copy the `https://xxxx.trycloudflare.com` URL from **Cell 6** and paste it as `COLAB_API_URL` in your Vercel environment variables

> **Note:** Free Colab allows ~12h per session. After that, re-run the notebook.  
> For truly permanent hosting, use **Oracle Cloud Always Free** (24 GB ARM, free forever).

In [ ]:
# ── Cell 1: Anti-idle JavaScript ─────────────────────────────────────────────
# Runs in the browser to prevent Colab's inactivity disconnect.
# Click the Colab menu: Runtime → Run all  (then leave this tab open)
from IPython.display import display, Javascript

display(Javascript('''
// Click the "Connect" / "Reconnect" button every 60 seconds
function keepAlive() {
  // Try to click the reconnect button if it appears
  var reconnectBtn = document.querySelector('colab-connect-button');
  if (reconnectBtn) {
    reconnectBtn.shadowRoot.querySelector('#connect').click();
    console.log('[keep-alive] clicked reconnect');
  }

  // Also prevent the "You have been idle" dialog
  try {
    var okBtn = document.querySelector('.mdl-dialog__actions button:first-child');
    if (okBtn) {
      okBtn.click();
      console.log('[keep-alive] dismissed idle dialog');
    }
  } catch (e) {}
}

// Run every 60 seconds
setInterval(keepAlive, 60000);
console.log('[keep-alive] anti-idle JS started — pinging every 60s');
'''))
print('✅ Anti-idle JavaScript started — keep this tab open in your browser')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Colab already has torch+CUDA. We only install the missing packages.
import subprocess, sys

print('Installing backend dependencies...')
packages = [
    'fastapi==0.115.0',
    'uvicorn==0.30.0',
    'pyngrok',           # backup tunnel (ngrok)
    'transformers>=4.41.0,<4.52.0',
    'sentencepiece>=0.2.0',
    'sacremoses>=0.1.1',
    'sentence-transformers>=3.0.0,<4.0.0',
    'rapidfuzz>=3.9.1',
    'sacrebleu>=2.4.0',
    'huggingface_hub>=0.23.0',
    'openai>=1.50.0',
    'python-dotenv>=1.0.1',
    'python-multipart>=0.0.9',
    'python-docx>=1.1.0',
    'pypdf2>=3.0.1',
    'pdfplumber>=0.11.0',
    'Pillow>=10.0.0',
    'opencv-python-headless>=4.9.0',
    'easyocr',
    'psutil',
    'requests>=2.32.0',
    'pandas>=2.2.2',
    'scikit-learn>=1.5.0',
    'numpy>=1.26.0',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages,
    check=True
)

# Install cloudflared (provides the public tunnel, no account needed)
print('Installing cloudflared tunnel...')
subprocess.run([
    'wget', '-q', '-O', '/usr/local/bin/cloudflared',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

print('\n✅ All dependencies installed')

In [ ]:
# ── Cell 3: Clone repo and set environment ────────────────────────────────────
import os, subprocess

# ── FILL IN YOUR VALUES HERE ──────────────────────────────────────────────────
HF_TOKEN      = 'hf_YOUR_TOKEN_HERE'   # HuggingFace token — get from hf.co/settings/tokens
GITHUB_TOKEN  = ''                      # Optional — only needed for feedback sync
# ─────────────────────────────────────────────────────────────────────────────

BACKEND_DIR = '/content/backend'

if not os.path.exists(BACKEND_DIR):
    print('Cloning repository...')
    subprocess.run([
        'git', 'clone', '--depth=1',
        'https://github.com/K227-arch/TRANSLATOR.git',
        '/content/TRANSLATOR'
    ], check=True)
    os.symlink('/content/TRANSLATOR/lunyoro-translator/backend', BACKEND_DIR)
else:
    print('Repo already cloned — pulling latest...')
    subprocess.run(['git', '-C', '/content/TRANSLATOR', 'pull', '--ff-only'], check=True)

os.chdir(BACKEND_DIR)

# Set environment variables
os.environ.update({
    'HF_TOKEN':       HF_TOKEN,
    'HF_USERNAME':    'keithtwesigye',
    'DISABLE_NLLB':   '0',    # NLLB enabled — runs locally on T4 GPU
    'DISABLE_MARIAN': '1',    # Marian disabled — NLLB only
    'DISABLE_ONNX':   '1',    # use FP32 PyTorch (GPU)
    'CORS_ORIGINS':   (
        'http://localhost:3002,'
        'https://horizonx.kathay.tech,'
        'https://runyoro-rutooro-translator.vercel.app,'
        'https://frontend-six-phi-25.vercel.app'
    ),
    'LOG_LEVEL':      'INFO',
    'PORT':           '8000',
    'HISTORY_FILE':   '/content/history.json',
    'GITHUB_TOKEN':   GITHUB_TOKEN,
})

print(f'✅ Environment set — backend dir: {BACKEND_DIR}')
print(f'   CUDA available: {__import__("torch").cuda.is_available()}')

In [ ]:
# ── Cell 4: Download NLLB models + semantic index ─────────────────────────────
import os, subprocess
os.chdir('/content/backend')

print('Downloading NLLB models + semantic search index...')
print('(~5 GB total — takes 3-5 min on Colab)')

result = subprocess.run(
    ['python', 'download_models.py', '--nllb-only'],
    capture_output=False
)

if result.returncode == 0:
    print('\n✅ Models downloaded successfully')
else:
    print('\n⚠️  Some models may have failed — check output above')

# Download the pre-built translation index from HF dataset
# (avoids rebuilding it from scratch which takes 10+ minutes)
index_path = '/content/backend/model/translation_index.pkl'
if not os.path.exists(index_path):
    print('\nDownloading translation index...')
    try:
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(
            repo_id='keithtwesigye/lunyoro-dataset',
            filename='translation_index.pkl',
            repo_type='dataset',
            local_dir='/content/backend/model'
        )
        print(f'✅ Translation index downloaded: {downloaded}')
    except Exception as e:
        print(f'⚠️  Could not download index ({e}) — building locally...')
        subprocess.run(['python', 'build_index.py'])
else:
    print(f'✅ Translation index already exists')

In [ ]:
# ── Cell 5: Start the backend server ─────────────────────────────────────────
import subprocess, threading, time, os
os.chdir('/content/backend')

server_process = None

def start_server():
    global server_process
    server_process = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'main:app',
         '--host', '0.0.0.0', '--port', '8000', '--workers', '1'],
        cwd='/content/backend',
    )
    return server_process

print('Starting uvicorn server on port 8000...')
t = threading.Thread(target=start_server, daemon=True)
t.start()

# Wait for server to be ready
import urllib.request
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=3)
        print(f'\n✅ Server is up! (took {(i+1)*2}s)')
        break
    except:
        print(f'  Waiting... {(i+1)*2}s', end='\r')
else:
    print('\n⚠️  Server took a while — NLLB may still be loading in background')

In [ ]:
# ── Cell 6: Start cloudflared tunnel + print public URL ──────────────────────
import subprocess, threading, time, re

tunnel_url = None
tunnel_process = None

def start_tunnel():
    global tunnel_url, tunnel_process
    tunnel_process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in tunnel_process.stdout:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

print('Starting cloudflared tunnel...')
t = threading.Thread(target=start_tunnel, daemon=True)
t.start()

# Wait for tunnel URL
for i in range(30):
    time.sleep(2)
    if tunnel_url:
        break

if tunnel_url:
    print(f'''
{'='*60}
✅  PUBLIC API URL (add to Vercel env vars):

   {tunnel_url}

   Set this as NEXT_PUBLIC_API_URL in your Vercel project:
   https://vercel.com/your-project/settings/environment-variables

   Health check:   {tunnel_url}/health
   API docs:       {tunnel_url}/docs
   System info:    {tunnel_url}/system-info
{'='*60}
''')
else:
    print('⚠️  Tunnel URL not detected — check cloudflared output above')

In [ ]:
# ── Cell 7: 24/7 Keep-alive loop ─────────────────────────────────────────────
# This cell runs FOREVER — it keeps the Colab session alive by:
#   1. Pinging the local server every 4 minutes (prevents Colab idle timeout)
#   2. Restarting the tunnel if it drops
#   3. Restarting the server if it crashes
#   4. Logging status every 10 minutes
#
# Leave this running. The session will still expire after ~12h on free Colab.

import time, urllib.request, subprocess, threading, re
from datetime import datetime

PING_INTERVAL  = 240   # 4 minutes — ping before Colab's ~5 min idle timeout
STATUS_INTERVAL = 600  # 10 minutes — print status

last_status = time.time()
ping_count  = 0
restart_count = 0

def _get_tunnel_url():
    """Re-parse tunnel URL from a new tunnel process."""
    global tunnel_url, tunnel_process
    if tunnel_process:
        try: tunnel_process.kill()
        except: pass
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    tunnel_process = proc
    for line in proc.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            return tunnel_url
    return None

def _server_alive():
    try:
        urllib.request.urlopen('http://localhost:8000/ping', timeout=5)
        return True
    except:
        return False

def _restart_server():
    global server_process, restart_count
    restart_count += 1
    print(f'[{datetime.now().strftime("%H:%M")}] ⚠️  Server down — restarting (attempt {restart_count})')
    try:
        if server_process: server_process.kill()
    except: pass
    import os
    os.chdir('/content/backend')
    server_process = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'main:app',
         '--host', '0.0.0.0', '--port', '8000', '--workers', '1'],
        cwd='/content/backend'
    )
    time.sleep(15)  # give it time to load models

print(f'🔄 Keep-alive loop started — pinging every {PING_INTERVAL//60} min')
print(f'   Current tunnel: {tunnel_url}')
print(f'   Ctrl+C or interrupt the cell to stop\n')

while True:
    time.sleep(PING_INTERVAL)
    ping_count += 1
    now = datetime.now().strftime('%H:%M:%S')

    # 1. Check server health
    if not _server_alive():
        _restart_server()

    # 2. Ping to prevent idle
    try:
        urllib.request.urlopen('http://localhost:8000/ping', timeout=5)
    except Exception as e:
        print(f'[{now}] ⚠️  Ping failed: {e}')

    # 3. Check tunnel
    if tunnel_process and tunnel_process.poll() is not None:
        print(f'[{now}] 🔁 Tunnel died — restarting...')
        new_url = _get_tunnel_url()
        if new_url:
            print(f'[{now}] ✅ New tunnel URL: {new_url}')
            print(f'       → Update NEXT_PUBLIC_API_URL in Vercel to: {new_url}')

    # 4. Periodic status
    if time.time() - last_status >= STATUS_INTERVAL:
        last_status = time.time()
        try:
            import json
            with urllib.request.urlopen('http://localhost:8000/system-info', timeout=5) as r:
                info = json.loads(r.read())
            nllb = info.get('nllb_en2lun', False)
            marian = info.get('marian_en2lun', False)
            gpu = info.get('gpu_available', False)
            print(f'[{now}] ✅ Status — NLLB:{nllb} Marian:{marian} GPU:{gpu} pings:{ping_count} restarts:{restart_count}')
            print(f'           Tunnel: {tunnel_url}')
        except Exception as e:
            print(f'[{now}] ⚠️  Status check failed: {e}')

## After getting the tunnel URL

1. Copy the `https://xxxx.trycloudflare.com` URL from Cell 6
2. Go to your Vercel project → **Settings → Environment Variables**
3. Set `NEXT_PUBLIC_API_URL` = `https://xxxx.trycloudflare.com`
4. Redeploy the frontend (or trigger a redeployment)

The tunnel URL **changes every time** you restart the notebook.  
To avoid updating Vercel each time, use a **custom domain with Cloudflare** or upgrade to **ngrok** (free account gives a static URL).

### Test it
```bash
curl https://xxxx.trycloudflare.com/health
curl -X POST https://xxxx.trycloudflare.com/translate \
     -H 'Content-Type: application/json' \
     -d '{"text":"Good morning"}'
```